# 🚢 Titanic Dataset — Exploratory Data Analysis (EDA)
**Internship Project 1**  
**Name:** Soha | **Department:** CSE (AI & ML) | **Institution:** AVN Institute of Engineering and Technology (JNTUH)

---


## Step 1 — Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 130, "axes.titlesize": 13})

df = sns.load_dataset('titanic')

print(f"Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\nMissing Values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nData Types:")
print(df.dtypes)
df.head()

## Step 2 — Handle Missing Values & Clean Data

In [ ]:
# Fix missing values
df['age'].fillna(df['age'].median(), inplace=True)
df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)
df['embark_town'].fillna(df['embark_town'].mode()[0], inplace=True)
df.drop(columns=['deck'], inplace=True)

# Create age buckets
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 12, 18, 35, 60, 100],
    labels=['Child (0-12)', 'Teen (13-18)', 'Young Adult (19-35)', 'Adult (36-60)', 'Senior (61+)']
)

print("✅ Cleaned! Remaining missing values:", df.isnull().sum().sum())
df.describe()

## Step 3 — Survival Rate Analysis

In [ ]:
overall = df['survived'].mean() * 100
print(f"Overall Survival Rate: {overall:.1f}%")

print("\nBy Sex:")
print((df.groupby('sex')['survived'].mean() * 100).round(1))

print("\nBy Class:")
print((df.groupby('pclass')['survived'].mean() * 100).round(1))

print("\nBy Age Group:")
print((df.groupby('age_group', observed=False)['survived'].mean() * 100).round(1))

print("\nBy Sex × Class:")
pivot = df.pivot_table(values='survived', index='sex', columns='pclass', aggfunc='mean') * 100
print(pivot.round(1))

## Step 4 — Visualizations
### Chart 1: Overview Dashboard

In [ ]:
COLORS = {
    'survived': '#2ecc71', 'not_survived': '#e74c3c',
    'male': '#3498db', 'female': '#e91e8c',
    'class': ['#1abc9c', '#f39c12', '#e74c3c'],
    'age': ['#9b59b6', '#3498db', '#2ecc71', '#f39c12', '#e74c3c'],
}
overall = df['survived'].mean() * 100
sex_surv = df.groupby('sex')['survived'].mean() * 100
class_surv = df.groupby('pclass')['survived'].mean() * 100
age_surv = df.groupby('age_group', observed=False)['survived'].mean() * 100

fig = plt.figure(figsize=(16, 10))
fig.suptitle('Titanic EDA — Overview Dashboard', fontsize=16, fontweight='bold')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Overall count
ax1 = fig.add_subplot(gs[0, 0])
counts = df['survived'].value_counts()
bars = ax1.bar(['Did Not Survive', 'Survived'], counts.values,
               color=[COLORS['not_survived'], COLORS['survived']], width=0.5, edgecolor='white')
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8, str(val), ha='center', fontweight='bold')
ax1.set_title('Overall Survival Count'); ax1.set_ylabel('Passengers'); ax1.set_ylim(0, 650)

# By sex
ax2 = fig.add_subplot(gs[0, 1])
sex_vals = sex_surv.reset_index()
bars2 = ax2.bar(sex_vals['sex'].str.capitalize(), sex_vals['survived'],
                color=[COLORS['male'], COLORS['female']], width=0.5, edgecolor='white')
for bar, val in zip(bars2, sex_vals['survived']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.1f}%', ha='center', fontweight='bold')
ax2.set_title('Survival Rate by Sex'); ax2.set_ylabel('Survival Rate (%)'); ax2.set_ylim(0, 100)
ax2.axhline(overall, color='gray', linestyle='--', linewidth=1, label=f'Overall {overall:.1f}%'); ax2.legend(fontsize=9)

# By class
ax3 = fig.add_subplot(gs[0, 2])
class_vals = class_surv.reset_index()
bars3 = ax3.bar([f'Class {c}' for c in class_vals['pclass']], class_vals['survived'],
                color=COLORS['class'], width=0.5, edgecolor='white')
for bar, val in zip(bars3, class_vals['survived']):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.1f}%', ha='center', fontweight='bold')
ax3.set_title('Survival Rate by Class'); ax3.set_ylabel('Survival Rate (%)'); ax3.set_ylim(0, 100)
ax3.axhline(overall, color='gray', linestyle='--', linewidth=1); 

# By age group
ax4 = fig.add_subplot(gs[1, 0])
age_vals = age_surv.reset_index()
bars4 = ax4.bar(age_vals['age_group'].astype(str), age_vals['survived'],
                color=COLORS['age'], width=0.6, edgecolor='white')
for bar, val in zip(bars4, age_vals['survived']):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.1f}%', ha='center', fontweight='bold', fontsize=8)
ax4.set_title('Survival Rate by Age Group'); ax4.set_ylabel('Survival Rate (%)'); ax4.set_ylim(0, 100)
ax4.tick_params(axis='x', labelsize=7)

# By embarkation
ax5 = fig.add_subplot(gs[1, 1])
emb_surv = df.groupby('embark_town')['survived'].mean() * 100
bars5 = ax5.bar(emb_surv.index, emb_surv.values, color=['#16a085','#8e44ad','#d35400'], width=0.5, edgecolor='white')
for bar, val in zip(bars5, emb_surv.values):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.1f}%', ha='center', fontweight='bold')
ax5.set_title('Survival by Embarkation'); ax5.set_ylabel('Survival Rate (%)'); ax5.set_ylim(0, 100)

# Class pie
ax6 = fig.add_subplot(gs[1, 2])
class_counts = df['pclass'].value_counts().sort_index()
ax6.pie(class_counts.values, labels=[f'Class {c}' for c in class_counts.index],
        autopct='%1.1f%%', colors=COLORS['class'], startangle=90,
        wedgeprops={'edgecolor':'white','linewidth':2})
ax6.set_title('Passenger Class Distribution')

plt.savefig('fig1_overview_dashboard.png', bbox_inches='tight', dpi=130)
plt.show()
print("Chart 1 saved!")

### Chart 2: Age & Distribution Analysis

In [ ]:
fig2, axes = plt.subplots(1, 3, figsize=(16, 5))
fig2.suptitle('Titanic EDA — Age & Distribution Analysis', fontsize=14, fontweight='bold')

sns.violinplot(x='survived', y='age', data=df,
               palette=[COLORS['not_survived'], COLORS['survived']], inner='box', ax=axes[0])
axes[0].set_xticks([0,1]); axes[0].set_xticklabels(['Did Not Survive','Survived'])
axes[0].set_title('Age Distribution by Survival\n(Violin Plot)'); axes[0].set_ylabel('Age')

sns.boxplot(x='pclass', y='age', hue='survived', data=df,
            palette=[COLORS['not_survived'], COLORS['survived']], ax=axes[1])
axes[1].set_title('Age by Class & Survival\n(Box Plot)')
axes[1].set_xlabel('Passenger Class'); axes[1].set_ylabel('Age')
axes[1].legend(title='Survived', labels=['No','Yes'])

axes[2].hist(df[df['survived']==0]['age'], bins=25, alpha=0.7, color=COLORS['not_survived'], label='Did Not Survive', edgecolor='white')
axes[2].hist(df[df['survived']==1]['age'], bins=25, alpha=0.7, color=COLORS['survived'], label='Survived', edgecolor='white')
axes[2].set_title('Age Distribution Histogram'); axes[2].set_xlabel('Age'); axes[2].set_ylabel('Count'); axes[2].legend()

plt.tight_layout()
plt.savefig('fig2_age_distribution.png', bbox_inches='tight', dpi=130)
plt.show()
print("Chart 2 saved!")

### Chart 3: Heatmap Analysis

In [ ]:
fig3, axes = plt.subplots(1, 2, figsize=(14, 5))
fig3.suptitle('Titanic EDA — Heatmap Analysis', fontsize=14, fontweight='bold')

corr = df.select_dtypes(include=np.number).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[0], linewidths=0.5)
axes[0].set_title('Correlation Heatmap')

pivot = df.pivot_table(values='survived', index='sex', columns='pclass', aggfunc='mean') * 100
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', ax=axes[1], linewidths=0.5,
            vmin=0, vmax=100, cbar_kws={'label': 'Survival Rate (%)'})
axes[1].set_title('Survival Rate (%)\nSex × Passenger Class'); axes[1].set_xlabel('Passenger Class')

plt.tight_layout()
plt.savefig('fig3_heatmaps.png', bbox_inches='tight', dpi=130)
plt.show()
print("Chart 3 saved!")

### Chart 4: Grouped Survival Analysis

In [ ]:
fig4, axes = plt.subplots(1, 2, figsize=(14, 5))
fig4.suptitle('Titanic EDA — Grouped Survival Analysis', fontsize=14, fontweight='bold')

sex_class = df.groupby(['pclass','sex'])['survived'].mean() * 100
sex_class = sex_class.reset_index()
x = np.arange(3); width = 0.35
male_vals = sex_class[sex_class['sex']=='male']['survived'].values
female_vals = sex_class[sex_class['sex']=='female']['survived'].values
axes[0].bar(x-width/2, male_vals, width, label='Male', color=COLORS['male'], edgecolor='white')
axes[0].bar(x+width/2, female_vals, width, label='Female', color=COLORS['female'], edgecolor='white')
for i,(m,f) in enumerate(zip(male_vals, female_vals)):
    axes[0].text(i-width/2, m+1, f'{m:.0f}%', ha='center', fontsize=9, fontweight='bold')
    axes[0].text(i+width/2, f+1, f'{f:.0f}%', ha='center', fontsize=9, fontweight='bold')
axes[0].set_xticks(x); axes[0].set_xticklabels(['1st Class','2nd Class','3rd Class'])
axes[0].set_ylabel('Survival Rate (%)'); axes[0].set_title('Survival Rate by Sex & Class')
axes[0].legend(); axes[0].set_ylim(0, 110)

age_counts = df.groupby(['age_group','survived'], observed=False).size().unstack()
age_counts.columns = ['Did Not Survive','Survived']
age_counts.index = age_counts.index.astype(str)
age_counts.plot(kind='bar', stacked=True, ax=axes[1],
                color=[COLORS['not_survived'], COLORS['survived']], edgecolor='white', width=0.6)
axes[1].set_title('Count by Age Group (Stacked)'); axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Number of Passengers'); axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('fig4_grouped_analysis.png', bbox_inches='tight', dpi=130)
plt.show()
print("Chart 4 saved!")

## Step 5 — Insight Report

---

### 📋 Key Findings from Titanic EDA

- **Insight 1 — Gender was the strongest survival predictor.**  
  Female passengers had a **74.2% survival rate** vs only **18.9%** for males, directly reflecting the "women and children first" evacuation protocol.

- **Insight 2 — Passenger class strongly influenced survival odds.**  
  1st class: **63%** | 2nd class: **47%** | 3rd class: **24%**. Wealthier passengers had upper-deck cabins with faster lifeboat access, creating a socioeconomic survival bias.

- **Insight 3 — Children had the highest survival among age groups.**  
  Passengers aged 0–12 survived at **~58%**, while Seniors (61+) had the lowest at **~23%**, reflecting both evacuation priority and physical limitations.

- **Insight 4 — The gender-class interaction revealed extreme disparity.**  
  1st class females survived at **96.8%**, while 3rd class males survived at only **13.5%** — an 83 percentage point gap showing the compounding effect of gender and wealth.

- **Insight 5 — Missing data required careful handling.**  
  ~20% of age values were missing (imputed with median = 28.0), and the 'deck' column (77% missing) was dropped entirely. Proper cleaning was essential before any meaningful analysis could begin.

---
*Dataset: 891 passengers | 15 original features | Overall Survival Rate: 38.4%*
